# Tutorial 4 — Primeira rede neural com dados sociodemográficos

**Nível:** Intermediário · **Tempo:** 35 min

**Pergunta:** Uma rede neural simples consegue aprender uma relação não linear em dados populacionais sintéticos?

> Dados didáticos sintéticos, não constituem evidência empírica.

## Passo 1 — Bibliotecas

A rede será treinada apenas em CPU, sem instalação de frameworks complexos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

## Passo 2 — Produzir uma relação não linear

O resultado binário é artificial e serve somente para visualizar relações não lineares.

In [ ]:
rng=np.random.default_rng(22)
n=2500
idade=rng.uniform(18,80,n)
renda=rng.uniform(500,6000,n)
X=np.column_stack([idade,renda])
score=-2.2+0.002*(idade-48)**2+0.00000016*(renda-3200)**2-0.001*(idade-50)*(renda-3200)
p=1/(1+np.exp(-np.clip(score,-10,10)))
y=rng.binomial(1,p)
X_train,X_test,y_train,y_test=train_test_split(X,y,stratify=y,test_size=.25,random_state=42)
print("Proporção da classe 1:",y.mean().round(3))

## Passo 3 — Treinar baseline e rede

O modelo de referência ajuda a verificar se a complexidade traz benefício.

In [ ]:
models={"Logística":make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),"Rede neural":make_pipeline(StandardScaler(),MLPClassifier(hidden_layer_sizes=(20,10),max_iter=600,early_stopping=True,random_state=42))}
for m in models.values():m.fit(X_train,y_train)

## Passo 4 — Comparar resultados

Resultados dependem do processo gerador, das escolhas de treinamento e da amostra.

In [ ]:
for nome,m in models.items():
    probs=m.predict_proba(X_test)[:,1]
    print(nome,"| ROC AUC:",round(roc_auc_score(y_test,probs),3))

## Passo 5 — Visualizar superfície de predição

A superfície é aprendida a partir de dados sintéticos, não de fenômenos reais.

In [ ]:
xx,yy=np.meshgrid(np.linspace(18,80,90),np.linspace(500,6000,90))
grade=np.column_stack([xx.ravel(),yy.ravel()])
probs=models["Rede neural"].predict_proba(grade)[:,1].reshape(xx.shape)
plt.figure(figsize=(8,5))
plt.contourf(xx,yy,probs,levels=12,cmap="viridis")
plt.xlabel("Idade (fictícia)");plt.ylabel("Renda (fictícia)");plt.colorbar(label="Probabilidade estimada")
plt.tight_layout();plt.show()

## Agora é sua vez

Altere hidden_layer_sizes para (5,) e (40,20). Compare AUC e superfícies. Observe se mais neurônios necessariamente melhoram o resultado.

## Limitações e cuidados

A rede não indica causalidade; validação temporal/geográfica e considerações de equidade seriam necessárias em estudos reais.